<h1 style='background:#FF008C; font-size:36px; color:white; padding-top:25px;'><center> <b> Tweets Classification with BERT </b> </center> </h1>

![](http://miro.medium.com/max/688/0*B8VDCnh8qBnwUuM4)

<h2 style='background:#FF6F00; color:white; padding:15px 0;'> Importing Required Libraries && tensorflow_text is must for Loading Preprocessor from Tensorflow hub. </h2>

In [1]:
!pip install tensorflow_text
import tensorflow_text as text

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.7/511.7 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.7/438.7 kB 33.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 21.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 40.2 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 2.6.0
    Uninstalling keras-2.6.0:
      Successfully uninstalled keras-2.6.0
  Attempting uninstall: tensorflow-estimator
    Found existing installation: tensorflow-estimator 2.6.0
    Uninstalling tensorflow-estimator-2.6.0:
      Successfully uninstalled tensorflow-estimator-2.6.0
  Attem

In [2]:
import re
import string
import numpy as np
import pandas as pd

from bs4 import BeautifulSoup

import plotly.express as px

import tensorflow as tf
import tensorflow_hub as hub

![](http://miro.medium.com/max/1200/1*fnfos3Z_xWdK9GdxX_pGxQ.jpeg)

 <h1 style='color:#ff8300; font-size:40px;'> <center> Bidirectional Encoder Representations from Transformers </center> </h1>

<div style='font-size:24px;'>
    
> BERT provides dense vector representations for natural language by using a deep, pre-trained neural network with the Transformer architecture. 

> Along with the Encoder, There is need to load the Bert Preprocessor which take cares of all of the text processing automatically that is being directly feed to the network.    
    
> ***BERT is the Model which is being used by Google in the Google Search Engine.***
    
> It is being trained on millions of wikipedia and other popular websites texts.
    
> Tensorflow provide API to load these Pretrained Models and Layers.
    
</div>

In [3]:
bert_preprocessor = hub.KerasLayer('https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3')
bert_encoder = hub.KerasLayer('https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4')

<h2 style='background:#FF6F00; color:white; padding:15px 0;'> Configuring TPU or GPU</h2>

![](https://miro.medium.com/max/640/1*Cf7HHO-ozFddK_qWT1U_vg.png)

In [4]:
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver() 
    print('Running on TPU ', tpu.master())
except ValueError:
    tpu = None

if tpu:
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.experimental.TPUStrategy(tpu)
else:
    strategy = tf.distribute.get_strategy() 

print("REPLICAS: ", strategy.num_replicas_in_sync)

REPLICAS:  1


<h2 style='background:#FF6F00; color:white; padding:15px 0;'> Setting up Dataset using Cloud Bucket (for TPU) ☁️ </h2>

In [5]:
from kaggle_datasets import KaggleDatasets
GCS_DS_PATH = KaggleDatasets().get_gcs_path('nlp-getting-started')
print(GCS_DS_PATH) 

gs://kds-8a3f6e6c53106f5ab37057131fe727ddc4e08411e5a33b8e3e247751


In [6]:
AUTO = tf.data.experimental.AUTOTUNE
BATCH_SIZE = 16 * strategy.num_replicas_in_sync
TRAINING_FILENAMES = tf.io.gfile.glob(GCS_DS_PATH + '/train.csv')

2022-07-07 15:46:24.525400: W tensorflow/core/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Couldn't resolve host name', error details: Could not resolve host: metadata".


In [7]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
raw_data = pd.read_csv(TRAINING_FILENAMES[0])

In [8]:
raw_data.head().style.set_table_styles(
    [{'selector': 'tr:hover',
      'props': [('background-color', 'green')]}]
).set_properties(**{
    'font-size': '15pt',})

,id,keyword,location,text,target
0,1,nan,nan,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,4,nan,nan,Forest fire near La Ronge Sask. Canada,1
2,5,nan,nan,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected,1
3,6,nan,nan,"13,000 people receive #wildfires evacuation orders in California",1
4,7,nan,nan,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,1


In [9]:
raw_data.isnull().sum()

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

In [10]:
custom_short_dict = {'AFAIK':'As Far As I Know',
'AFK':'Away From Keyboard',
'ASAP':'As Soon As Possible',
'ATK':'At The Keyboard',
'ATM':'At The Moment',
'A3':'Anytime, Anywhere, Anyplace',
'BAK':'Back At Keyboard',
'BBL':'Be Back Later',
'BBS':'Be Back Soon',
'BFN':'Bye For Now',
'B4N':'Bye For Now',
'BRB':'Be Right Back',
'BRT':'Be Right There',
'BTW':'By The Way',
'B4':'Before',
'B4N':'Bye For Now',
'CU':'See You',
'CUL8R':'See You Later',
'CYA':'See You',
'FAQ':'Frequently Asked Questions',
'FC':'Fingers Crossed',
'FWIW':'For What It is Worth',
'FYI':'For Your Information',
'GAL':'Get A Life',
'GG':'Good Game',
'GN':'Good Night',
'GMTA':'Great Minds Think Alike',
'GR8':'Great!',
'G9':'Genius',
'IC':'I See',
'ICQ':'I Seek you',
'ILU':'I Love You',
'IMHO':'In My Honest',
'IMO':'In My Opinion',
'IOW':'In Other Words',
'IRL':'In Real Life',
'KISS':'Keep It Simple, Stupid',
'LDR':'Long Distance Relationship',
'LMAO':'Laugh My Ass',
'LOL':'Laughing Out Loud',
'LTNS':'Long Time No See',
'L8R':'Later',
'MTE':'My Thoughts Exactly',
'M8':'Mate',
'NRN':'No Reply Necessary',
'OIC':'Oh I See',
'PITA':'Pain In The Ass',
'PRT':'Party',
'PRW':'Parents Are Watching',
'ROFL':'Rolling On The Floor Laughing',
'ROFLOL':'Rolling On The Floor Laughing Out Loud',
'ROTFLMAO':'Rolling On The Floor Laughing My Ass',
'SK8':'Skate',
'STATS':'Your sex and age',
'ASL':'Age, Sex, Location',
'THX':'Thank You',
'TTFN':'Ta-Ta For Now!',
'TTYL':'Talk To You Later',
'U':'You',
'U2':'You Too',
'U4E':'Yours For Ever',
'WB':'Welcome Back',
'WTF':'What The Fuck',
'WTG':'Way To Go!',
'WUF':'Where Are You From?',
'W8':'Wait'}

In [11]:
raw_data.replace(regex=custom_short_dict, inplace=True)

**The Below code show some king of basic Preprocessing for Natural Language Preprocessing.**

The Preprocessing steps are as follows:
> 1. Removing Stopwords.
> 2. Removing Punctuations & Symbols.
> 3. Removing Http Code using Beautiful Soup.

In [12]:
stopwords = ["a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at",
             "be", "because", "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do",
             "does", "doing", "down", "during", "each", "few", "for", "from", "further", "had", "has", "have", "having",
             "he", "hed", "hes", "her", "here", "heres", "hers", "herself", "him", "himself", "his", "how",
             "hows", "i", "id", "ill", "im", "ive", "if", "in", "into", "is", "it", "its", "itself",
             "lets", "me", "more", "most", "my", "myself", "nor", "of", "on", "once", "only", "or", "other", "ought",
             "our", "ours", "ourselves", "out", "over", "own", "same", "she", "shed", "shell", "shes", "should",
             "so", "some", "such", "than", "that", "thats", "the", "their", "theirs", "them", "themselves", "then",
             "there", "theres", "these", "they", "theyd", "theyll", "theyre", "theyve", "this", "those", "through",
             "to", "too", "under", "until", "up", "very", "was", "we", "wed", "well", "were", "weve", "were",
             "what", "whats", "when", "whens", "where", "wheres", "which", "while", "who", "whos", "whom", "why",
             "whys", "with", "would", "you", "youd", "youll", "youre", "youve", "your", "yours", "yourself",
             "yourselves"]

table = str.maketrans('', '', string.punctuation)

In [13]:
def sentence_replication(sentence):
    sentence = re.sub(r'http\S+', '', sentence)
    sentence = sentence.replace(",", " , ")
    sentence = sentence.replace(".", " . ")
    sentence = sentence.replace("-", " - ")
    sentence = sentence.replace("/", " / ")
    sentence = sentence.replace("\\", " \\ ")
    sentence = sentence.replace("0", " ")
    sentence = sentence.replace("1", " ")
    sentence = sentence.replace("2", " ")
    sentence = sentence.replace("3", " ")
    sentence = sentence.replace("4", " ")
    sentence = sentence.replace("5", " ")
    sentence = sentence.replace("6", " ")
    sentence = sentence.replace("7", " ")
    sentence = sentence.replace("8", " ")
    sentence = sentence.replace("9", " ")
    soup = BeautifulSoup(sentence)
    sentence = soup.getText()
    return sentence

def PreProcessing(df, training=True):
    
    df['keyword'] = df['keyword'].fillna('')
    df['location'] = df['location'].fillna('')
    
    SENTENCE = []
    LABELS = []
    
    for idx in df.index:
        sentence = raw_data.iloc[idx, 1].lower()
        sentence += ' '
        sentence += raw_data.iloc[idx, 2].lower()
        sentence += ' '
        sentence += raw_data.iloc[idx, 3].lower()
        sentence = sentence_replication(sentence)
        words = sentence.split()
        filtered_sentence = ''
        for word in words:
            word = word.translate(table)
            if word not in stopwords:
                filtered_sentence = filtered_sentence + word + ' '
        SENTENCE.append(filtered_sentence)
        
        if training:
            label = raw_data.iloc[idx, 4]
            LABELS.append(label)
            
    if training:
        return np.array(SENTENCE), np.array(LABELS)
    
    return np.array(SENTENCE)

In [14]:
X, Y = PreProcessing(raw_data)

/opt/conda/lib/python3.7/site-packages/bs4/__init__.py:439: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  MarkupResemblesLocatorWarning


In [15]:
def build_model():
    text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text_input')
    preprocessed_text = bert_preprocessor(text_input)
    output = bert_encoder(preprocessed_text)
    o = tf.keras.layers.Dropout(0.1, name='Dropout')(output['pooled_output'])
    o = tf.keras.layers.Dense(units=1, activation='sigmoid')(o)
    
    model = tf.keras.Model(inputs=[text_input], outputs=[o])
    
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    return model

> <h2 style='color:#ff6f00'> Model summary shows that we need to only train the Parameter of our DNN i.e. 768 weights + 1 bias = 769. </h2>

In [16]:
tweet_model = build_model()
tweet_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 text_input (InputLayer)        [(None,)]            0           []                               
                                                                                                  
 keras_layer (KerasLayer)       {'input_word_ids':   0           ['text_input[0][0]']             
                                (None, 128),                                                      
                                 'input_mask': (Non                                               
                                e, 128),                                                          
                                 'input_type_ids':                                                
                                (None, 128)}                                                  

In [17]:
train_dataset = tf.data.Dataset.from_tensor_slices((X, Y))
train_dataset = train_dataset.shuffle(64).batch(BATCH_SIZE)
train_dataset = train_dataset.cache()
train_dataset = train_dataset.prefetch(AUTO)

In [18]:
model_history = tweet_model.fit(train_dataset, epochs=20)

Epoch 1/20
476/476 [==============================] - 60s 106ms/step - loss: 0.6408 - accuracy: 0.6415
Epoch 2/20
476/476 [==============================] - 50s 106ms/step - loss: 0.5790 - accuracy: 0.7071
Epoch 3/20
476/476 [==============================] - 51s 106ms/step - loss: 0.5581 - accuracy: 0.7282
Epoch 4/20
476/476 [==============================] - 51s 107ms/step - loss: 0.5489 - accuracy: 0.7364
Epoch 5/20
476/476 [==============================] - 58s 121ms/step - loss: 0.5394 - accuracy: 0.7436
Epoch 6/20
476/476 [==============================] - 59s 124ms/step - loss: 0.5331 - accuracy: 0.7510
Epoch 7/20
476/476 [==============================] - 56s 117ms/step - loss: 0.5281 - accuracy: 0.7517
Epoch 8/20
476/476 [==============================] - 53s 111ms/step - loss: 0.5256 - accuracy: 0.7534
Epoch 9/20
476/476 [==============================] - 52s 109ms/step - loss: 0.5207 - accuracy: 0.7548
Epoch 10/20
476/476 [==============================] - 52s 110ms/step - l

<h2 style='background:#FF6F00; color:white; padding:15px 0;'> Visualization of Model Training Accuracy & Loss </h2>

> **Unavailable**

In [19]:
#hist_df = pd.DataFrame(model_history.history)
#fig = px.line(hist_df)
#fig.update_layout(template='plotly_dark', width=1200, title='Metrics')
#fig.show()

<h2 style='background:#FF6F00; color:white; padding:15px 0;'> Test Submission </h2>

In [20]:
test_df = pd.read_csv('../input/nlp-getting-started/test.csv')
test_df.replace(regex=custom_short_dict, inplace=True)
id_col = test_df.iloc[:, 0]
x_test = PreProcessing(test_df, training=False)

In [21]:
pred = tweet_model.predict(x_test)
pred = np.where(pred>=0.5, 1, 0)
pred = pred.reshape(-1)

102/102 [==============================] - 22s 211ms/step


In [22]:
df = {'Id':id_col, 'target':pred}
df = pd.DataFrame(df)
df.to_csv('Submission.csv', index=False)

<div style='background:#FF6F00;'>
    <h1 style='padding:15px 0 0 0; color:white'> <center> Thanks for Reading 😀 </center></h1>
</div>

<h2 style='padding:15px 0 0 0; background:#FF00D6; color:white'> <center> 🪧 Work in Progress 🪧 </center></h2>